# WoodelfHD

Runs **WoodelfHD using GPU** across all datasets and task types
in the `woodelfhd_depth_sweep_experiment`. Use the T4 runtime type with High-RAM avaiable (50GB RAM).
This notebook is **independent** and can run in parallel with the other method notebooks.

### What this notebook does
1. Mounts Google Drive (results are saved there after each mission)
2. Clones `treebranchmarks` repo
3. Installs all dependencies
4. Runs `woodelfhd_depth_sweep_experiment --method woodelf_hd_gpu`
5. Writes partial results to Drive as `woodelf_hd_gpu.json`

### Datasets (all download automatically)
| Dataset | Source |
|---------|--------|
| Fraud Detection | Google Drive parquet (~200 MB) |
| HIGGS | Google Drive parquet |
| KDD Cup (Intrusion Detection) | Google Drive parquet |
| California Housing | sklearn builtin |

> **Runtime estimate:** Expect up to 2 hours on a T4 Colab GPU runtime with High-RAM (50GB RAM).

In [ ]:
# ── Step 1: Mount Google Drive ──────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ── Step 2: Configure paths ──────────────────────────────────────────────────
# Edit DRIVE_FOLDER to point to your preferred Google Drive folder.
# All other paths are derived from it.
import pathlib

DRIVE_FOLDER = pathlib.Path('/content/drive/MyDrive/ShapResearch/HighDepth/treebranchmark_experiments/results_jsons')
DRIVE_FOLDER.mkdir(parents=True, exist_ok=True)

DRIVE_RESULT_PATH = DRIVE_FOLDER / 'woodelf_hd_gpu.json'
print(f'Results will be saved to: {DRIVE_RESULT_PATH}')

Results will be saved to: /content/drive/MyDrive/ShapResearch/HighDepth/treebranchmark_experiments/results_jsons/woodelf_hd_gpu.json


In [ ]:
# ── Step 3: Clone repository ───────────────────────────────────────────────
TREEBRANCHMARKS_URL = 'https://github.com/ron-wettenstein/TreeBranchMarks.git'

!git clone {TREEBRANCHMARKS_URL} /content/treebranchmarks

Cloning into '/content/treebranchmarks'...
remote: Enumerating objects: 384, done.
remote: Counting objects: 100% (384/384), done.
remote: Compressing objects: 100% (231/231), done.
remote: Total 384 (delta 247), reused 277 (delta 143), pack-reused 0 (from 0)
Receiving objects: 100% (384/384), 222.04 KiB | 980.00 KiB/s, done.
Resolving deltas: 100% (247/247), done.


In [ ]:
# ── Step 4: Install packages ─────────────────────────────────────────────────
# woodelf_explainer must be installed before treebranchmarks (it is listed
# as a dependency in treebranchmarks/pyproject.toml).

!pip install woodelf_explainer

!pip install -q -e /content/treebranchmarks

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.4/41.4 kB 4.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 66.4 MB/s eta 0:00:00
  Building editable for treebranchmarks (pyproject.toml) ... done


In [ ]:
# ── Step 5: Restore method cache from a previous interrupted run ─────────────
# The framework writes the method cache file to Drive after every approach result.
# On restart, copy it back to the local cache directory so the experiment
# skips already-completed entries and only runs what is still missing.
import shutil, pathlib

cache_dir = pathlib.Path('/content/treebranchmarks/cache/method_results/woodelfhd_depth_sweep_experiment')
cache_dir.mkdir(parents=True, exist_ok=True)
local_cache_file = cache_dir / 'woodelf_hd_gpu.json'

if DRIVE_RESULT_PATH.exists() and not local_cache_file.exists():
    shutil.copy(DRIVE_RESULT_PATH, local_cache_file)
    print(f'Restored method cache ({DRIVE_RESULT_PATH.stat().st_size // 1024} KB)')
else:
    print('No method cache to restore — starting fresh.')

Restored method cache (46 KB)


In [ ]:
# ── Step 6: Run the experiment (WoodelfHD only) ───────────────────────────────
# -u   : unbuffered output so progress prints appear in real time
# --method woodelf_hd : only the WoodelfHD approach is timed
# --result_location   : dual-write to Drive after every completed mission

%cd /content/treebranchmarks

!python -u -m benchmarks.woodelfhd_depth_sweep_experiment \
    --method woodelf_hd_gpu \
    --result_location "{DRIVE_RESULT_PATH}"

/content/treebranchmarks

Mission: fraud_detection PD SHAP sweep_D
  dataset   : fraud_detection
  D_values  : [6, 9, 12, 15, 18, 21]
  n=118108  m=0
  tasks     : ['Path-Dependent SHAP']

  > D=6  n=118108  m=0
  [approach:WoodelfHD GPU] CACHED=15.049s

  > D=9  n=118108  m=0
  [approach:WoodelfHD GPU] CACHED=41.610s

  > D=12  n=118108  m=0
  [approach:WoodelfHD GPU] CACHED=98.236s

  > D=15  n=118108  m=0
  [approach:WoodelfHD GPU] CACHED=182.714s

  > D=18  n=118108  m=0
  [approach:WoodelfHD GPU] CACHED=390.511s

  > D=21  n=118108  m=0
  [approach:WoodelfHD GPU] CACHED=2782.667s

Mission: fraud_detection BG SHAP sweep_D
  dataset   : fraud_detection
  D_values  : [6, 9, 12, 15, 18, 21]
  n=118108  m=472432
  tasks     : ['Background SHAP']

  > D=6  n=118108  m=472432
  [approach:WoodelfHD GPU] CACHED=15.060s

  > D=9  n=118108  m=472432
  [approach:WoodelfHD GPU] CACHED=46.325s

  > D=12  n=118108  m=472432
  [approach:WoodelfHD GPU] CACHED=105.859s

  > D=15  n=118108  m=472432

In [ ]:
# ── Step 7: Verify output ────────────────────────────────────────────────────
import json

with open(DRIVE_RESULT_PATH) as f:
    cache = json.load(f)

print(f'Entries in method cache: {len(cache)}')
if cache:
    sample = next(iter(cache.values()))
    print(f'Sample entry: {sample["_label"]}  →  {sample["running_time"]:.3f}s')
print(f'\nFile saved to: {DRIVE_RESULT_PATH}')

Entries in method cache: 94
Sample entry: Path-Dependent SHAP n=118108 m=0 D=6 T=100  →  15.049s

File saved to: /content/drive/MyDrive/ShapResearch/HighDepth/treebranchmark_experiments/results_jsons/woodelf_hd_gpu.json


# Old Runs

In [ ]:
# ── Step 6: Run the experiment (WoodelfHD only) ───────────────────────────────
# -u   : unbuffered output so progress prints appear in real time
# --method woodelf_hd : only the WoodelfHD approach is timed
# --result_location   : dual-write to Drive after every completed mission

%cd /content/treebranchmarks

!python -u -m benchmarks.woodelfhd_depth_sweep_experiment \
    --method woodelf_hd_gpu \
    --result_location "{DRIVE_RESULT_PATH}"

/content/treebranchmarks

Mission: fraud_detection PD SHAP sweep_D
  dataset   : fraud_detection
  D_values  : [6, 9, 12, 15, 18, 21]
  n=118108  m=0
  tasks     : ['Path-Dependent SHAP']
[dataset:fraud_detection] Cache miss — downloading and preprocessing.
Downloading...
From: https://drive.google.com/uc?id=1A1Qdtron9XtZ6h85uNdaFCprUkH5KA5P
To: /content/treebranchmarks/cache/datasets/fraud_detection/raw/data.parquet
100% 69.6M/69.6M [00:01<00:00, 53.6MB/s]
[dataset:fraud_detection] Cached 590540 rows × 397 features.

  > D=6  n=118108  m=0
[model:lightgbm] Training.
[model:lightgbm] Trained in 13.17s — T=100, D=6, L=32.0, F=397
Preprocessing the trees and computing SHAP: 100% 100/100 [00:14<00:00,  6.93it/s]
M time: 0.31 sec, s time: 11.03 sec (f prepare time: 4.814283609390259)
  [approach:WoodelfHD GPU] mean=15.049s

  > D=9  n=118108  m=0
[model:lightgbm] Training.
[model:lightgbm] Trained in 15.62s — T=100, D=9, L=82.8, F=397
Preprocessing the trees and computing SHAP: 100% 100/10